# Karplus-Strong


Using delay-line inital conditions to "pluck" the string consisting of random numbers, or white noise. The inital shape of the string is obtained by adding the upper and lower delay lines, we also apply a low-pass filter by taking the average of adjacent samples to obtain:  $$y(n) = 0.5(y(n-N) + y(n-N-1))$$ where $N$ is the sample length. Pitch is determined by $$ f \approx fs / (N+0.5)$$ where $fs$ is the sample rate, set to $44100$ by standard. Without the 2-point average, a raw looped buffer is exactly $ fs/ N$.

In [ ]:
%matplotlib inline
import numpy as np, wave, struct
import matplotlib.pyplot as plt
from IPython.display import display, Audio

In [ ]:
fs = 44100 #sample rate
N = 100 # delay-line length
duration = 2.0

In [ ]:
noise = np.random.uniform(-1,1, N) #White noise initial state
output = np.zeros(int(fs * duration))
output[0] = noise[0]
idx = 0
last = 0

for i in range(int(fs * duration)):
    output[i] = noise[idx]
    x = noise[idx]
    noise[idx] = 0.5 * (x + last)
    last = x 
    idx = (idx + 1) % N


In [ ]:
def plot_waveform_spectrum(output, name = "output"):
    fig, ax = plt.subplots(2, 1, figsize=(10, 6))
    
    t_ms = np.arange(len(output)) / fs * 1000
    ax[0].plot(t_ms[:250], output[:250])
    ax[0].set_title(f"waveform - {name}")
    ax[0].set_xlabel('ms'); ax[0].set_ylabel('amp')
    
    X = np.abs(np.fft.rfft(output * np.hanning(len(output))))
    f = np.fft.rfftfreq(len(output), 1/fs)
    ax[1].plot(f, 20*np.log10(X/np.max(X) + 1e-9))
    ax[1].set_xlim(0, 6000); ax[1].set_ylim(-80, 2)
    ax[1].set_title(f"waveform - {name}"); ax[1].set_xlabel('Hz'); ax[1].set_ylabel('dB')
    
    plt.tight_layout(); plt.show()

In [ ]:
plot_waveform_spectrum(output)

In [ ]:
Audio(output, rate = fs)

## The Pluck
The pluck is a triangular initial state at a point on the string. Each harmonic $n$ has a fixzed spatil shape along the string with nodes at $ x = L * k/n$. 

In [ ]:
def pluck_string(beta=0.5, N = 100, duration = 2.0, fs = 44100, rho = 1.0 ):
    p = int(beta *N) 
    increasing = np.linspace(0, 1, p) 
    decreasing = np.linspace(1,0, N-p)
    noise = np.concatenate((increasing,decreasing))
                   
    output = np.zeros(int(fs * duration))
    output[0] = noise[0]
    idx = 0
    last = 0
    
    for i in range(int(fs * duration)):
        output[i] = noise[idx]
        x = noise[idx]
        noise[idx] = 0.5 * (x + last) * rho
        last = x 
        idx = (idx + 1) % N

    return output

In [ ]:
output1 = pluck_string(0.5)
output2 = pluck_string(0.1)


In [ ]:
plot_waveform_spectrum(output1, "output1")
plot_waveform_spectrum(output2, "output2")

In [ ]:
print("beta = 0.5 (center)"); display(Audio(output1, rate=fs))   # center pluck — hollow
print("beta = 0.1 (bridge)"); display(Audio(output2, rate=fs))   # bridge pluck — bright

## The Body 
Currently, we have just a string that displaces only a little of air so it is quiet. In an actual guitar, the string excites the air in the box of the guitar which resonates out the sound hole. We will use Finite Element to capture the characteristics of the body. 

**Finite Difference on a 1D stiff bar**
A string resists displacement by tension, a bar resists change by storing energy in its stiffness. This is a checkpoint in implimenting FE but will also be revisited in future explorations into percussion (xylophones, marimba, etc.)
$$ \text{string (tension): } \partial ^2 y / \partial t^2 = c^2 \cdot \partial ^2 y / \partial x^2 $$
$$ \text{bar (stiffness): } \partial ^2 y / \partial t^2 = -\kappa^2 \cdot \partial ^4 y / \partial x^4 $$
where $\kappa = EI / (\rho A)$ and $E$ is the Young's modulus, $I$ is the how the cross-ssection resists bending, $\rho$ is density and $A$ is area. The fourth derivative is the physical signature of bending and it makes the bar dispersive, high-frequency bending waves travels faster than low-frequency waves. Thus, stiff bars are inharmonic and its mode frequency ratios depend only on the boundary conditions.

By applying central difference twice: 
$$y''''_i = \frac{ y_{i-2} - 4y_{i-1} + 6y_i - 4y_{i+1} + y_{i+2} }{ h^4 }$$ 
The stiffness matrix $\mathcal{k}$ is coefficients of each point and the mass matrix $M$ is the identity matrix with equal point masses. So the bar's equation of motion is 
$$ ÿ  = -(\kappa ^2 / h^4) \cdot \mathcal{K} \cdot y $$
and the modes are the eigenvectors of $\mathcal{K}$:
$$ \mathcal{K} \phi = \lambda \phi $$
$$ \lambda = (\omega^2 h^4) / K^4$$
and mode frequency is 
$$ \omega = \sqrt{(\kappa ^ 2 / h^4) \cdot \lambda_i} $$ 

In [ ]:
import scipy

In [ ]:
M = 200 # number of points
K = np.zeros((M,M)) #stiffness matrix

#pinned (simply-supported) boundary rows to get the clean ratio 1 : 4 : 9 : 16 ...
K[0,0]=5;   K[0,1]=-4;   K[0,2]=1                       # first free point
K[1,0]=-4;  K[1,1]=6;    K[1,2]=-4;   K[1,3]=1          # second free point
K[M-2,M-1]=-4; K[M-2,M-2]=6; K[M-2,M-3]=-4; K[M-2,M-4]=1   # symmetric, far end
K[M-1,M-1]=5;  K[M-1,M-2]=-4; K[M-1,M-3]=1

#

#populating the internal points of the stiffness matrix with (1,-4,6,-4,1) along the diagonals of cols. i-2 ... i+2
for i in range(2,M-2,1):
    K[i][i-2] = 1
    K[i][i-1] = -4
    K[i][i] = 6
    K[i][i+1] = -4
    K[i][i+2] = 1

#Eigenproblem

evals, evecs = scipy.linalg.eigh(K) 
print("Expected ratios: 1, 4, 9, 16, 25")
np.sqrt(evals[:5]) / np.sqrt(evals[0]) 

In [ ]:
x = np.arange(M)
fig, ax = plt.subplots(4, 1, figsize=(9,8), sharex = True)
for k in range(4):
    ax[k].plot(x ,evecs[:,k])
    ax[k].axhline(0,color='gray',lw=0.5) # the flat rest position
    ax[k].set_ylabel(f'mode {k+1}')
ax[-1].set_xlabel('position along bar')
plt.tight_layout()
plt.show()

In [ ]:
fs = 44100
duration = 2.0
t = np.arange(int(fs*duration))/fs
n_modes = 10
s = 15 #strike point
f_0 = 220.0

In [ ]:
def bar_strike(decay = 300, s =15, n_modes =10):
    freqs = f_0 * np.sqrt(evals) / np.sqrt(evals[0])
    
    y = np.zeros_like(t)
    for k in range(n_modes):
        amp = evecs[s,k] # how hard the strike excites mode k
        tau = 2.0 / (1+freqs[k]/decay) # higher modes decay faster, tune to taste
        y += amp * np.sin(2*np.pi*freqs[k]*t) * np.exp(-t/tau)
    
    y = y /np.max(np.abs(y)) # normalize
    return y

In [ ]:
bar_sound = bar_strike()

In [ ]:
plot_waveform_spectrum(bar_sound, "1D Bar Strike")

In [ ]:
print('s = 15')
display(Audio(bar_sound , rate = fs))
bar_sound_2 = bar_strike(300,30)
print ('s = 30')
display(Audio(bar_sound_2 , rate = fs))
bar_sound_3 = bar_strike(300,5)
print ('s = 5')
display(Audio(bar_sound_3 , rate = fs))

**Ideal 2D Plate**
The 2D plate is governed by:
$$\partial ^2 w / \partial t^2 = (-D/\rho h) \cdot \nabla ^4 w $$
where $w(x,y)$ is the displacement of each point of the sheet and $\nabla^4$  is the biharmonic operator. The total pull of the forces are:
$$\frac{\partial^2 w}{\partial x^2} + \frac{\partial^2 w}{\partial y^2} = \frac{w[i+1,j] + w[i,j+1] - 4w[i,j] + w[i-1,j] + w[i,j-1]}{h^2} $$

In [ ]:
Nx = Ny = 20
N = Nx * Ny 
def idx(i,j): return i * Ny + j # flattening

#2D Laplacian, -4 on the point, + 1 on each neightbor, using pinned boundary so neighbors off the grid are skipped
L = np.zeros((N,N))
for i in range(Nx):
    for j in range(Ny):
        p = idx(i,j)
        L[p, p] = -4
        if i > 0: L[p, idx(i-1,j)] = 1
        if i < Nx-1: L[p, idx(i+1,j)] = 1
        if j > 0: L[p,idx(i,j-1)] = 1
        if j < Ny-1: L[p,idx(i,j+1)] = 1
#Plate = Laplacian squared
K = L @ L

# modes
evals, evecs = scipy.linalg.eigh(K)
print(np.sqrt(evals[:8]) / np.sqrt(evals[0])) # Expected: ~[1,2.5,2.5,4,5,5,6.5,6.5]

In [ ]:
def plate_strike(decay = 300, sx = 10, sy = 10, lx = 10, ly = 10, n_modes = 10): #sx, sy represent strike point. lx,ly represent listening point 
    freqs = f_0 * np.sqrt(evals) / np.sqrt(evals[0])
    y = np.zeros_like(t)
    for k in range(n_modes):
        amp = evecs[idx(sx,sy),k] * evecs[idx(lx,ly),k] # how hard the strike excites mode k
        tau = 2.0 / (1+freqs[k]/decay) # higher modes decay faster, tune to taste
        y += amp * np.sin(2*np.pi*freqs[k]*t) * np.exp(-t/tau)
    
    y = y /np.max(np.abs(y)) # normalize
    return y

In [ ]:
plate_sound_1 = plate_strike()

In [ ]:
plot_waveform_spectrum(plate_sound_1, "2D Plate Strike")

In [ ]:
print('s = 10,10')
display(Audio(bar_sound , rate = fs))
plate_strike_2 = plate_strike(300,5,10, 5, 10)
print('s = 5,10')
display(Audio(plate_strike_2 , rate = fs))
plate_strike_3 = plate_strike(300,5,20, 10,10)
print('s = 5,20')
display(Audio(plate_strike_3 , rate = fs))

**Real Block, Guitar Body**  

Right now there is a pair of $2.5$ which are degenerate because the isotropic biharmonic is 
$$ \frac{\partial^2 w}{\partial x^2} + \frac{\partial^2 w}{\partial x^2} = \frac{\partial w^4}{\partial x^4} + 2 \frac{\partial^4 w}{\partial x^2 \partial y^2 } + \frac{\partial^4 w}{\partial y^4}  $$
So x,y enters identically and direction is lost. 
Wood is orthotropic meaning it is stiffer along the grain than across it. So the two directions get different bending rigidites $D_x \neq D_y$ so the plate operator becomes:
$$ D_x \partial^4 _x + 2 H \partial^2 _x \partial ^2 _y + D_y \partial ^4_y $$
This breaks the symmetry and the degenerate pair splits. The mode that bends more along the stiff direction rises in pitch faster than its partner. Squaring the anisotropic laplacian gives the orthotropic plate:
$$(a_x \partial _{xx} + a_y\partial_{yy})^2 = a^2_x \partial_x^4 + 2a_xa_y \partial^2_x \partial^2_y + a^2_y \partial^4_y $$
with $D_x = ax^2$ and $D_y = ay^2$ and cross term $H = \sqrt{D_x D_y}$ 

In [ ]:
def orthotropic_plate(ax = 4 , ay = 1, resolution = 20): #ax = 1, ay =1 should reproduce ideal
    Nx = Ny = resolution
    N = Nx * Ny 
    def idx(i,j): return i * Ny + j # flattening
    
    #2D Anisotropic Laplacian
    L_a = np.zeros((N,N))
    for i in range(Nx):
        for j in range(Ny):
            p = idx(i,j)
            L_a[p, p] = -(2*ax + 2*ay)
            if i > 0: L_a[p, idx(i-1,j)] = ax
            if i < Nx-1: L_a[p, idx(i+1,j)] = ax
            if j > 0: L_a[p,idx(i,j-1)] = ay
            if j < Ny-1: L_a[p,idx(i,j+1)] = ay
    #Plate = Laplacian squared
    K = L_a @ L_a
    
    # modes
    evals, evecs = scipy.linalg.eigh(K)
    print(np.sqrt(evals[:8]) / np.sqrt(evals[0])) 
    return K, evals, evecs

In [ ]:
plate1, plate1Evals, plate1Evecs = orthotropic_plate()

In [ ]:
def real_plate_strike(K, evals, evecs, Q = 20, sx = 10, sy = 10, lx = 10, ly = 10, n_modes = 10): #sx, sy represent strike point. lx,ly represent listening point 
    # Recover the grid size from the eigenvector count instead of the global Ny=20.
    # evecs has res*res rows, laid out as i*res + j, so res = sqrt(#rows).
    res = int(round(np.sqrt(evecs.shape[0])))

    def idx(i, j):                      # local idx, matches how orthotropic_plate flattened
        i = min(max(i, 0), res - 1)     # clamp so a point never falls off the grid
        j = min(max(j, 0), res - 1)
        return i * res + j

    freqs = f_0 * np.sqrt(evals) / np.sqrt(evals[0])
    y = np.zeros_like(t)
    for k in range(n_modes):
        amp = evecs[idx(sx,sy),k] * evecs[idx(lx,ly),k] # how hard the strike excites mode k
        tau = Q/(np.pi*freqs[k]) #decay
        y += amp * np.sin(2*np.pi*freqs[k]*t) * np.exp(-t/tau)
    
    y = y /np.max(np.abs(y)) # normalize
    return y

In [ ]:
real_plate_sound_1 = real_plate_strike(plate1, plate1Evals, plate1Evecs)

In [ ]:
plot_waveform_spectrum(real_plate_sound_1, "2D Real Plate Strike")

In [ ]:
display(Audio(real_plate_sound_1 , rate = fs))

**A guitar is a linear chain of blocks**  

The string is an oscillator and the body is a filters the colors the stringss sound with its own resonances. Both are linear and time-invariant so the whole body can be folded into a single fixed operation. To pass the string through the body we need to convolve:
$$ \text{guitar} = \text{string} \circledast \text{body}_{\text{IR}} $$

In [ ]:
def sustain_duration(rho=0.99, N=100):
    T60 = -6.91 * N / (fs * np.log(rho))
    dur = 1.2 * T60
    return np.arange(int(fs * dur)) / fs, dur

t,duration = sustain_duration()   # <-- bind it to the global t everything reads

In [ ]:
guitar1,guitar1Evals,guitar1Evecs = orthotropic_plate(4, 1, 40)
body_IR = real_plate_strike(guitar1, guitar1Evals, guitar1Evecs)
string = pluck_string(rho = 0.99, duration = duration)
guitar1_sound = scipy.signal.fftconvolve(string, body_IR) #note that convoltion is one-way when a real bridge is two-way

M = 2000
guitar1_sound [-M:] *= np.linspace(1,0,M) #Ramp the last M samples to zero for a smooth fade

In [ ]:
plot_waveform_spectrum(guitar1_sound, "Convoluted Body-String System")

In [ ]:
display(Audio(guitar1_sound,rate=fs))

Reference: https://ccrma.stanford.edu/~jos/pasp/Karplus_Strong_Algorithm.html

**Sequencer-voice Model**  

To demostrate this guitar and future fictional instruments we used a sequencer-voice workflow to play a normal scale. A normal scale is defined by:
$$ f_k = f_{root} \cdot 2 ^ {k/12}$$

In [ ]:
def make_scale(root_hz, semitones):
    return root_hz * 2 ** (np.array(semitones) / 12)

In [ ]:
def guitar_voice(f, body_IR, rho = 0.99, beta = 0.5):
    N = round(fs / f- 0.5) # solving for line length by inverting karplus-strong pitch law f = f_s/ (N + 0.5)
    s = pluck_string(beta = beta, rho = rho, N=N)
    note = scipy.signal.fftconvolve(s, body_IR) # same as before, we convolve the body resonance with string resonance
    M = 2000
    note[-M:] *= np.linspace(1,0,M)
    return note

In [ ]:
def play_sequence(freqs, voice, dt =0.4, fs=44100): 
    notes = [voice(f) for f in freqs]
    onsets = [round(i * dt * fs) for i in range(len(freqs))]
    total = max(o + len(n) for o, n in zip(onsets, notes))
    out = np.zeros(total)
    for o, n in zip(onsets, notes): 
        out[o:o+len(n)] += n
    return out / np.max(np.abs(out))

In [ ]:
display(Audio(pluck_string(N=round(fs/220 - 0.5)), rate=fs))  # low
display(Audio(pluck_string(N=round(fs/440 - 0.5)), rate=fs))  # octave up

**Guitar Major Scale**

In [ ]:
body_IR = real_plate_strike(guitar1, guitar1Evals, guitar1Evecs, n_modes = 30)

In [ ]:
scale = make_scale(220, [0,2,4,5,7,9,11,12])
print(scale)

for f in make_scale(220, [0,2,4,5,7,9,11,12]):
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.99)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))

**Chord** 

In [ ]:
scale = make_scale(220, [1,3,5])
print(scale)

for f in make_scale(220, [1,3,5]):
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.99, beta = 0.13)
song = play_sequence(scale, guitar, dt = 0)
display(Audio(song, rate=fs))

**Some Ficitonal Bodies** (still using pluck string)

High Stiffness Ratio

In [ ]:
stiff,stiffEvals,stiffEvecs = orthotropic_plate(100000, 1, 40)
stiff_body_IR = real_plate_strike(stiff, stiffEvals, stiffEvecs)

In [ ]:
scale = make_scale(220, [0,2,4,5,7,9,11,12])
print(scale)

for f in make_scale(220, [0,2,4,5,7,9,11,12]):
    print(f, round(fs/f - 0.5))

ideal = lambda f: guitar_voice(f, stiff_body_IR, rho=0.995)
song = play_sequence(scale, ideal)
display(Audio(song, rate=fs))

High Beta, Same Rho

In [ ]:
scale = make_scale(220, [0,2,4,5,7,9,11,12])
print(scale)

for f in make_scale(220, [0,2,4,5,7,9,11,12]):
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.99, beta = 1)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))

Low Beta, Same Rho

In [ ]:
scale = make_scale(220, [0,2,4,5,7,9,11,12])
print(scale)

for f in make_scale(220, [0,2,4,5,7,9,11,12]):
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.99, beta = 0.1)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))

High Beta, Long Sustain (High Rho)

In [ ]:
scale = make_scale(220, [0,2,4,5,7,9,11,12])
print(scale)

for f in make_scale(220, [0,2,4,5,7,9,11,12]):
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.999, beta = 1)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))

Low Beta, Short Sustain (Low Rho)

In [ ]:
scale = make_scale(220, [0,2,4,5,7,9,11,12])
print(scale)

for f in make_scale(220, [0,2,4,5,7,9,11,12]):
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.9, beta = 1)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))

Changing the semitones

In [ ]:
scale = make_scale(220, [0,1,2,3,4,5,6,7,8,9,10,11,12])
print(scale)

for f in scale:
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.99)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))

Changing the timbre (Thud)

In [ ]:
body_IR = real_plate_strike(guitar1, guitar1Evals, guitar1Evecs, Q = 5, n_modes = 30)
scale = make_scale(220, [0,1,2,3,4,5,6,7,8,9,10,11,12])
print(scale)

for f in scale:
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.99)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))

Changing the timbre (Bell)


In [ ]:
body_IR = real_plate_strike(guitar1, guitar1Evals, guitar1Evecs, Q = 500, n_modes = 30)
scale = make_scale(220, [0,1,2,3,4,5,6,7,8,9,10,11,12])
print(scale)

for f in scale:
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.99)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))

## Fractional Delay  
Currently we use $N = round(fs/ f- 0.5)$ to determine delay line length which determines pitch, however, this shaves due to rounding. Let $D$ be the delay needed to achieve $f$. Then 
$$ D = \frac{fs}{f} - 0.5 $$ and let $frac = D -N \in [0,1)$ and $ N = \lfloor D \rfloor$. We take the weighted blend of neighboring samples by linear interpolation
$$ y = (1-frac) \cdot d[N] + frac \cdot d[N+1] $$ 

In [ ]:
def pluck_string(beta=0.5, f = 220 , duration = 2.0, fs = 44100, rho = 1.0 ):

    D = fs/f - 0.5 #Fractional delay variables
    N = int(D) 
    frac = D - N
    
    p = int(beta *N) 
    increasing = np.linspace(0, 1, p) 
    decreasing = np.linspace(1,0, N-p)
    pluck = np.concatenate((increasing,decreasing))
                   
    L = N + 2 # longer buffer than max delay
    buf = np.zeros(L)
    buf[:N] = pluck
    out = np.zeros(int(fs*duration))
    w = 0 #index for reading delay line
    last = 0.0
    
    for i in range(len(out)):
        r0 = (w - N) % L # tap at delay N
        r1 = (w - N - 1) % L # tap one sample older at N -1
        y = (1-frac) * buf[r0] + frac*buf[r1]  #fractional read
        buf[w] = 0.5 * (y +last) * rho
        last = y 
        out[i] = y
        w = (w+1) % L
    return out

In [ ]:
def guitar_voice(f, body_IR, rho = 0.99, beta = 0.5):
    s = frac_delay_pluck_string(beta = beta, rho = rho, f=f)
    note = scipy.signal.fftconvolve(s, body_IR) # same as before, we convolve the body resonance with string resonance
    M = 2000
    note[-M:] *= np.linspace(1,0,M)
    return note

In [ ]:
body_IR = real_plate_strike(guitar1, guitar1Evals, guitar1Evecs, n_modes = 30)
scale = make_scale(220, [0,2,4,5,7,9,11,12])
print(scale)

for f in make_scale(220, [0,2,4,5,7,9,11,12]):
    print(f, round(fs/f - 0.5))

guitar = lambda f: guitar_voice(f, body_IR, rho=0.99)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))

## The Soundhole
The soundhole of a guitar is a helmholtz resonator.
$$f_H = \frac{c}{2\pi} \sqrt{\frac{A}{V * L_{eff}}} $$ 
with $c$ the speed of sound, $A$ the area of the soundhole, $V$ the cavity volume and $L_{eff}$ the effective neck length. For real guitars, this is about 90-110Hz, called the air resonance, which is the frequency at which air is breathed in and out the soundhole. Then we have multiple oscillators, the plates and the soundhole, coupled together. Coupled oscillators always repel so the three bare frequencies split into three new normal modes. 

**Top plate / Back Plate / Enclosed Air**  
We split the top into the top plate and back plate, then with the enclosed air we have a total of three oscillators. At low frequency, there are three degrees of freedom, the top plate flexing, the back plate flexing and the air moving in and out of the soundhole. They are linked by the enclosed cavity air acting as a shared spring.

**Three Body Eigenproblem**  
Each part has displacement (where air into the cavity is positive), an effective mass, a piston area, and its own stiffness. Let $M = diag(m_{top}, m_{back},m_{hole})$ be the mass matrix. And $K$ be the stiffness matrix defined by:
$$K = 
\begin{bmatrix}
m_{top} \omega_{top}^2 & 0 & 0 \\
0 & m_{back}\omega^2_{back} & 0 \\
0 & 0 & 0
\end{bmatrix} + \frac{\rho c^2}{V} aa^T
$$
Where $a$ is the area vector $a = [A_{top}, A_{back}, A_{hole}]$. Solving the eigenproblem nets three eigenvectors for how much the top plate, back plate, and air moves in that mode. The string drives the top 

In [ ]:
def three_oscillator_model(f_top=180, f_back=200, m_top=0.15, m_back=0.05,
                           Volume=0.012, hole_radius=0.041, hole_thickness=0.003,
                           top_area=0.18, back_area=0.013, rho=1.2, c=343):
    omega_top  = 2*np.pi*f_top
    omega_back = 2*np.pi*f_back
    hole_area  = np.pi * hole_radius**2
    L_eff      = hole_thickness + 1.2*hole_radius     # end corrections, both faces
    m_hole     = rho * hole_area * L_eff              # air-plug mass (~3e-4 kg)

    a = np.array([top_area, back_area, -hole_area])   # air leaves through hole
    M = np.diag([m_top, m_back, m_hole])
    K = np.diag([m_top*omega_top**2, m_back*omega_back**2, 0.0]) + (rho * c**2 / Volume) * np.outer(a, a)

    evals, evecs = scipy.linalg.eigh(K, M)
    freqs = np.sqrt(evals) / (2*np.pi)
    return freqs, evecs

In [ ]:
def modal_bank(freqs, amps, Qs, t):
    y = np.zeros_like(t)
    for f, a, Q in zip(freqs, amps, Qs):
        tau = Q / (np.pi * f)
        y += a * np.sin(2*np.pi*f*t) * np.exp(-t/tau)
    return y

In [ ]:
def build_body_IR(evals, evecs, idx, t, n_modes=30, Q=20,
                  sx=10, sy=10, lx=10, ly=10,
                  f_top=180, g_low=1.0, **osc):
    # higher plate modes (skip mode 0 = top fundamental, now handled by coupling)
    plate_f = f_top * np.sqrt(evals) / np.sqrt(evals[0])
    freqs_p, amps_p, Qs_p = [], [], []
    for k in range(1, n_modes):
        freqs_p.append(plate_f[k])
        amps_p.append(evecs[idx(sx, sy), k] * evecs[idx(lx, ly), k])
        Qs_p.append(Q)

    # three coupled low modes (top + back + air)
    freqs3, ev3 = three_oscillator_model(f_top=f_top, **osc)
    amps3 = [ev3[0, k] * (ev3[0, k] + ev3[2, k]) for k in range(3)]
    Qs3   = [30, 30, 15]

    # combine into one modal bank
    freqs = np.concatenate([freqs3, freqs_p])
    amps  = np.concatenate([g_low * np.array(amps3), amps_p])
    Qs    = np.concatenate([Qs3, Qs_p])

    y = modal_bank(freqs, amps, Qs, t)
    return y / np.max(np.abs(y))

In [ ]:
guitar1, guitar1Evals, guitar1Evecs = orthotropic_plate(4, 1, 40)
body_IR = build_body_IR(guitar1Evals, guitar1Evecs, idx, t,
                        n_modes=30, g_low=1.0)

# then, as before:
guitar = lambda f: frac_delay_guitar_voice(f, body_IR, rho=0.99)
song = play_sequence(scale, guitar)
display(Audio(song, rate=fs))